# 🤖 UPES Lost & Found — AI Matching Model
### Minor Project — B.Tech (Hons.) CSE, Semester 6
**University of Petroleum and Energy Studies**

---
This notebook demonstrates the complete ML pipeline used to match **lost** and **found** item reports:
1. Text-based similarity (TF-IDF + Cosine)
2. Image-based similarity (Color Histogram)
3. Combined scoring with weights
4. Model evaluation (Precision, Recall, F1)

## 📦 1. Install & Import Libraries

In [ ]:
# Install required packages (run once)
# !pip install scikit-learn pillow numpy pandas matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

print('✅ All libraries imported successfully')

## 🗄️ 2. Sample Dataset
We simulate real campus data with 20 lost + found pairs.

In [ ]:
# ── LOST ITEMS ──────────────────────────────────────────────────────────
lost_items = pd.DataFrame([
    {'id':'L001','name':'Black Nike Backpack',       'category':'Bags',       'location':'Library 2nd floor',    'description':'Black Nike backpack with laptop compartment, white swoosh, stickers on front'},
    {'id':'L002','name':'iPhone 14 Blue Case',       'category':'Electronics','location':'Canteen near windows', 'description':'iPhone 14 with blue cracked case, small dent on bottom corner'},
    {'id':'L003','name':'Honda Bike Keys',           'category':'Keys',       'location':'Parking Lot A',        'description':'Honda key with red keyring tag, small torch attached'},
    {'id':'L004','name':'Prescription Glasses',      'category':'Other',      'location':'Block A Room 204',     'description':'Black thin frame prescription glasses in brown leather case'},
    {'id':'L005','name':'Samsung Galaxy Watch',      'category':'Electronics','location':'Sports Complex',       'description':'Samsung galaxy watch black strap silver body, slightly scratched'},
    {'id':'L006','name':'Blue Adidas Hoodie',        'category':'Clothing',   'location':'Hostel Block 3',       'description':'Blue Adidas hoodie size XL has name Rahul written inside collar'},
    {'id':'L007','name':'Calculus Textbook',         'category':'Books',      'location':'Block B Room 105',     'description':'Engineering Calculus Sem 3 textbook lots of handwritten notes yellow cover'},
    {'id':'L008','name':'UPES Student ID Card',      'category':'Cards',      'location':'Library entry',        'description':'UPES student ID card CSE department with photo'},
    {'id':'L009','name':'Dell Laptop Charger',       'category':'Electronics','location':'Lab 3 Computer Lab',   'description':'Dell 65W charger black cable with brick adapter'},
    {'id':'L010','name':'Gold Earrings',             'category':'Jewellery',  'location':'Girls Hostel Common',  'description':'Small gold hoop earrings pair in a red velvet pouch'},
    {'id':'L011','name':'Red Titan Wristwatch',      'category':'Electronics','location':'Canteen ground floor', 'description':'Titan red dial watch with brown leather strap'},
    {'id':'L012','name':'Navy Blue Polo T-shirt',    'category':'Clothing',   'location':'Gym changing room',    'description':'Navy blue polo shirt size L Lacoste brand'},
    {'id':'L013','name':'Physics Lab Record',        'category':'Books',      'location':'Physics Lab Block C',  'description':'Physics practical record notebook blue cover with 500119640 written'},
    {'id':'L014','name':'Umbrella Black',            'category':'Other',      'location':'Admin Block reception','description':'Black automatic umbrella with curved handle slight tear on canopy'},
    {'id':'L015','name':'Silver Bracelet',           'category':'Jewellery',  'location':'Library reading room',  'description':'Thin silver bracelet with heart charm'},
    {'id':'L016','name':'MacBook Air Charger',       'category':'Electronics','location':'Seminar Hall',          'description':'Apple MagSafe charger white cable 60W for MacBook Air'},
    {'id':'L017','name':'Cricket Bat',              'category':'Sports',      'location':'Sports Ground',        'description':'SS cricket bat with red grip medium size'},
    {'id':'L018','name':'Green Water Bottle',        'category':'Other',      'location':'Block D cafeteria',    'description':'Dark green Milton water bottle 1 litre with UPES sticker'},
    {'id':'L019','name':'Wireless Earbuds',          'category':'Electronics','location':'Library study pods',   'description':'White wireless earbuds in charging case, brand boAt'},
    {'id':'L020','name':'Driving Licence',           'category':'Cards',      'location':'Parking area',         'description':'DL card with photo, Uttarakhand state, name Tilak Yadav'},
])

# ── FOUND ITEMS ─────────────────────────────────────────────────────────
found_items = pd.DataFrame([
    {'id':'F001','name':'Nike Backpack Black',        'category':'Bags',       'location':'Library near stairs',  'description':'Black nike bag with laptop inside, white logo on front'},
    {'id':'F002','name':'Phone with Blue Cover',     'category':'Electronics','location':'Canteen table 12',     'description':'iPhone in cracked blue cover found on canteen table'},
    {'id':'F003','name':'Keyring with Honda Key',    'category':'Keys',       'location':'Parking gate area',    'description':'Honda motorcycle key with red plastic keyring and torch'},
    {'id':'F004','name':'Spectacles with Case',      'category':'Other',      'location':'Block A corridor',     'description':'Prescription spectacles black thin frame inside brown case'},
    {'id':'F005','name':'Smartwatch Samsung',        'category':'Electronics','location':'Volleyball court',     'description':'Samsung watch black band found near sports area'},
    {'id':'F006','name':'Hoodie Blue Colour',        'category':'Clothing',   'location':'Common room hostel',   'description':'Blue hoodie XL Adidas brand'},
    {'id':'F007','name':'Engineering Maths Book',   'category':'Books',      'location':'Classroom B105',       'description':'Calculus book sem 3 with handwritten notes found under bench'},
    {'id':'F008','name':'College ID Card',           'category':'Cards',      'location':'Library main gate',    'description':'Student identity card UPES CSE branch'},
    {'id':'F009','name':'Laptop Charger Dell',       'category':'Electronics','location':'Computer lab near PC8','description':'Dell charger 65 watt black color found near computer'},
    {'id':'F010','name':'Gold Hoop Earrings',        'category':'Jewellery',  'location':'Hostel washroom',      'description':'Pair of small gold hoops in a red small pouch'},
    {'id':'F011','name':'Titan Watch Red Dial',      'category':'Electronics','location':'Canteen counter',      'description':'Titan watch red display brown leather band'},
    {'id':'F012','name':'Blue Polo Shirt',           'category':'Clothing',   'location':'Gym locker area',      'description':'Navy blue polo L size lacoste logo'},
    {'id':'F013','name':'Lab Record Notebook',       'category':'Books',      'location':'Physics lab table 4',  'description':'Practical notebook blue cover physics experiments'},
    {'id':'F014','name':'Black Umbrella',            'category':'Other',      'location':'Admin office entry',   'description':'Automatic black umbrella curved handle near reception'},
    {'id':'F015','name':'Bracelet Silver',           'category':'Jewellery',  'location':'Library 1st floor',    'description':'Thin silver bracelet with a small heart pendant'},
    {'id':'F016','name':'Apple Charger White',       'category':'Electronics','location':'Seminar hall chair row3','description':'White MacBook charger MagSafe 60W'},
    {'id':'F017','name':'Cricket Bat Red Grip',      'category':'Sports',     'location':'Pavilion ground',      'description':'SS bat cricket medium red handle'},
    {'id':'F018','name':'Water Bottle Green Milton', 'category':'Other',      'location':'D Block canteen table','description':'Green 1L milton bottle has UPES sticker'},
    {'id':'F019','name':'Earphones Wireless boAt',  'category':'Electronics','location':'Library pod 7',        'description':'boAt white wireless earbuds with white charging case'},
    {'id':'F020','name':'Driving Licence Card',      'category':'Cards',      'location':'Bike parking zone',    'description':'Driving licence Uttarakhand male photo found on ground'},
])

# Ground truth: L001 matches F001, L002 matches F002, ... (diagonal)
ground_truth = {f'L{str(i).zfill(3)}': f'F{str(i).zfill(3)}' for i in range(1,21)}

print(f'Lost items  : {len(lost_items)}')
print(f'Found items : {len(found_items)}')
print(f'Ground truth pairs: {len(ground_truth)}')
display(lost_items.head(5))

## 🧠 3. Text-Based Matching — TF-IDF + Cosine Similarity

In [ ]:
def combine_text(row):
    """Combine all text fields into one string for vectorization."""
    return f"{row['name']} {row['category']} {row['description']}".lower()

lost_texts  = lost_items.apply(combine_text, axis=1).tolist()
found_texts = found_items.apply(combine_text, axis=1).tolist()

# Build TF-IDF on combined corpus
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),    # unigrams + bigrams
    max_features=500,
    stop_words='english'
)
all_texts = lost_texts + found_texts
vectorizer.fit(all_texts)

lost_vecs  = vectorizer.transform(lost_texts)
found_vecs = vectorizer.transform(found_texts)

# Compute cosine similarity matrix (20 x 20)
text_sim_matrix = cosine_similarity(lost_vecs, found_vecs)
print('TF-IDF Similarity Matrix shape:', text_sim_matrix.shape)

# Visualize
plt.figure(figsize=(12,8))
sns.heatmap(text_sim_matrix,
            xticklabels=found_items['id'],
            yticklabels=lost_items['id'],
            cmap='YlOrRd', annot=True, fmt='.2f', linewidths=.3)
plt.title('Text Similarity Matrix (TF-IDF Cosine)', fontsize=14, fontweight='bold')
plt.xlabel('Found Items')
plt.ylabel('Lost Items')
plt.tight_layout()
plt.savefig('text_similarity_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nSaved: text_similarity_matrix.png')

## 🎨 4. Category Boost
Items in the same category get a score boost.

In [ ]:
CATEGORY_BOOST = 0.15  # Add 15% if categories match

def apply_category_boost(text_sim, lost_df, found_df):
    boosted = text_sim.copy()
    for i, lrow in lost_df.iterrows():
        for j, frow in found_df.iterrows():
            if lrow['category'].lower() == frow['category'].lower():
                boosted[i][j] = min(1.0, boosted[i][j] + CATEGORY_BOOST)
    return boosted

boosted_matrix = apply_category_boost(
    text_sim_matrix.copy(), lost_items.reset_index(), found_items.reset_index()
)
print('Category boost applied.')
print(f'Sample score before boost: {text_sim_matrix[0][0]:.3f}')
print(f'Sample score after  boost: {boosted_matrix[0][0]:.3f}')

## 📊 5. Model Evaluation — Precision, Recall, F1

In [ ]:
def evaluate_model(sim_matrix, lost_df, found_df, ground_truth, threshold=0.20):
    y_true, y_pred = [], []
    predictions = []
    
    for i, lrow in lost_df.iterrows():
        best_j    = np.argmax(sim_matrix[i])
        best_score = sim_matrix[i][best_j]
        pred_fid  = found_df.iloc[best_j]['id'] if best_score >= threshold else None
        true_fid  = ground_truth.get(lrow['id'])
        correct   = (pred_fid == true_fid)
        y_true.append(1)
        y_pred.append(1 if correct else 0)
        predictions.append({
            'Lost ID'    : lrow['id'],
            'Lost Item'  : lrow['name'],
            'Predicted'  : found_df.iloc[best_j]['name'] if best_score>=threshold else '(no match)',
            'True Match' : found_df[found_df['id']==true_fid]['name'].values[0] if true_fid else '?',
            'Score'      : round(best_score, 3),
            'Correct'    : '✅' if correct else '❌'
        })
    
    accuracy  = sum(y_pred) / len(y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall    = recall_score(y_true, y_pred, zero_division=0)
    f1        = f1_score(y_true, y_pred, zero_division=0)
    return accuracy, precision, recall, f1, pd.DataFrame(predictions)

# Evaluate text-only
acc_t, prec_t, rec_t, f1_t, pred_df_text = evaluate_model(
    text_sim_matrix, lost_items, found_items, ground_truth, threshold=0.20
)

# Evaluate with category boost
acc_b, prec_b, rec_b, f1_b, pred_df_boost = evaluate_model(
    boosted_matrix, lost_items, found_items, ground_truth, threshold=0.20
)

print('='*55)
print('          MODEL EVALUATION RESULTS')
print('='*55)
print(f'               Text-Only   Text+Category')
print(f'  Accuracy  :   {acc_t:.3f}        {acc_b:.3f}')
print(f'  Precision :   {prec_t:.3f}        {prec_b:.3f}')
print(f'  Recall    :   {rec_t:.3f}        {rec_b:.3f}')
print(f'  F1 Score  :   {f1_t:.3f}        {f1_b:.3f}')
print('='*55)

display(pred_df_boost)

## 📈 6. Score Distribution Plot

In [ ]:
# Top-1 scores for each lost item
top1_scores = [boosted_matrix[i].max() for i in range(len(lost_items))]
correct_matches = [1 if (np.argmax(boosted_matrix[i]) == i) else 0 for i in range(len(lost_items))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score distribution
colors = ['#2e7d32' if c==1 else '#c62828' for c in correct_matches]
bars = axes[0].bar(lost_items['id'], top1_scores, color=colors, edgecolor='white', linewidth=0.5)
axes[0].axhline(y=0.20, color='orange', linestyle='--', linewidth=1.5, label='Threshold (0.20)')
axes[0].set_title('Top-1 Match Score per Lost Item', fontweight='bold')
axes[0].set_xlabel('Lost Item ID')
axes[0].set_ylabel('Similarity Score')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()
# Legend patch
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2e7d32',label='Correct Match'), Patch(facecolor='#c62828',label='Wrong Match')]
axes[0].legend(handles=legend_elements + [plt.Line2D([0],[0],color='orange',linestyle='--',label='Threshold')])

# Category-wise performance
categories = lost_items['category'].unique()
cat_scores = []
for cat in categories:
    idxs = lost_items[lost_items['category']==cat].index.tolist()
    scores = [boosted_matrix[i].max() for i in idxs]
    cat_scores.append(np.mean(scores))

axes[1].barh(categories, cat_scores,
             color=['#5c35a0','#00bcd4','#2e7d32','#e65100','#c62828','#6a1b9a','#1565c0','#37474f'][:len(categories)])
axes[1].set_title('Average Match Score by Category', fontweight='bold')
axes[1].set_xlabel('Average Similarity Score')
axes[1].axvline(x=0.20, color='orange', linestyle='--', linewidth=1.5)

plt.suptitle('UPES Lost & Found — AI Match Model Performance', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_performance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: model_performance.png')

## 🔍 7. Threshold Analysis — Finding the Optimal Cut-off

In [ ]:
thresholds = np.arange(0.05, 0.95, 0.05)
metrics = []
for t in thresholds:
    acc, prec, rec, f1, _ = evaluate_model(boosted_matrix, lost_items, found_items, ground_truth, threshold=t)
    metrics.append({'threshold':t,'accuracy':acc,'precision':prec,'recall':rec,'f1':f1})

mdf = pd.DataFrame(metrics)

plt.figure(figsize=(10, 5))
plt.plot(mdf['threshold'], mdf['accuracy'],  label='Accuracy',  marker='o', linewidth=2)
plt.plot(mdf['threshold'], mdf['f1'],        label='F1 Score',  marker='s', linewidth=2)
plt.plot(mdf['threshold'], mdf['precision'], label='Precision', marker='^', linewidth=2, linestyle='--')
plt.plot(mdf['threshold'], mdf['recall'],    label='Recall',    marker='v', linewidth=2, linestyle='--')
best_t = mdf.loc[mdf['f1'].idxmax(), 'threshold']
plt.axvline(x=best_t, color='red', linestyle=':', linewidth=1.5, label=f'Best threshold = {best_t:.2f}')
plt.title('Model Metrics vs Decision Threshold', fontsize=13, fontweight='bold')
plt.xlabel('Threshold'); plt.ylabel('Score')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('threshold_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✅ Optimal threshold: {best_t:.2f}')
print(f'   Best F1 Score    : {mdf["f1"].max():.3f}')

## 🖼️ 8. Image Color Histogram Similarity (Demo)
Simulates what happens when photos are uploaded.

In [ ]:
import math

def color_hist_similarity(hist_a, hist_b):
    """Bhattacharyya coefficient between two normalized histograms."""
    return sum(math.sqrt(a * b) for a, b in zip(hist_a, hist_b))

# Simulate histograms for 3 bins (R, G, B dominance)
demo_items = [
    {'name': 'Black Nike Backpack (Lost)',  'hist': [0.05, 0.05, 0.90]},  # mostly dark
    {'name': 'Black Nike Bag (Found)',      'hist': [0.06, 0.05, 0.89]},  # very similar
    {'name': 'Blue Hoodie (Found)',         'hist': [0.10, 0.70, 0.20]},  # mostly blue
    {'name': 'Red Wallet (Found)',          'hist': [0.80, 0.10, 0.10]},  # mostly red
]

# Compare lost item (index 0) with all found items
lost_h  = demo_items[0]['hist']
results = []
for item in demo_items[1:]:
    sim = color_hist_similarity(lost_h, item['hist'])
    results.append({'Item': item['name'], 'Image Similarity': round(sim, 3)})

res_df = pd.DataFrame(results).sort_values('Image Similarity', ascending=False)

# Plot
plt.figure(figsize=(8, 4))
bars = plt.barh(res_df['Item'], res_df['Image Similarity'],
                color=['#2e7d32','#5c35a0','#c62828'])
plt.axvline(x=0.5, color='orange', linestyle='--', label='Threshold 0.5')
for bar, val in zip(bars, res_df['Image Similarity']):
    plt.text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=10)
plt.title('Image Color Similarity — Lost: Black Nike Backpack', fontweight='bold')
plt.xlabel('Bhattacharyya Similarity Score')
plt.xlim(0, 1.15)
plt.legend(); plt.tight_layout()
plt.savefig('image_similarity.png', dpi=120, bbox_inches='tight')
plt.show()
display(res_df)

## ⚖️ 9. Combined Score (Text + Image)

In [ ]:
# Demonstrate combined scoring formula
example = pd.DataFrame([
    {'Lost Item':'Black Nike Backpack','Found Item':'Nike Backpack Black',   'text_score':0.87,'img_score':0.94},
    {'Lost Item':'Black Nike Backpack','Found Item':'Blue Adidas Hoodie',    'text_score':0.12,'img_score':0.21},
    {'Lost Item':'Black Nike Backpack','Found Item':'Samsung Galaxy Watch',  'text_score':0.08,'img_score':0.18},
    {'Lost Item':'Black Nike Backpack','Found Item':'Calculus Textbook',     'text_score':0.05,'img_score':0.10},
])

TEXT_WEIGHT  = 0.60
IMAGE_WEIGHT = 0.40

example['combined_score'] = (
    example['text_score']  * TEXT_WEIGHT +
    example['img_score']   * IMAGE_WEIGHT
).round(3)

print('Combined Score Formula:')
print(f'  score = text_score × {TEXT_WEIGHT} + image_score × {IMAGE_WEIGHT}\n')

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(example))
w = 0.25
bars1 = ax.bar(x - w,     example['text_score'],     w, label='Text Score (TF-IDF)', color='#5c35a0')
bars2 = ax.bar(x,         example['img_score'],      w, label='Image Score',         color='#00bcd4')
bars3 = ax.bar(x + w,     example['combined_score'], w, label='Combined Score',      color='#2e7d32')
ax.axhline(y=0.20, color='red', linestyle='--', linewidth=1.2, label='Match Threshold')
ax.set_xticks(x)
ax.set_xticklabels(example['Found Item'], rotation=15, ha='right')
ax.set_ylabel('Similarity Score'); ax.set_ylim(0, 1.1)
ax.set_title('Combined Matching Score vs Individual Scores', fontweight='bold')
ax.legend(); plt.tight_layout()
plt.savefig('combined_scores.png', dpi=120, bbox_inches='tight')
plt.show()
display(example)

## ✅ 10. Summary & Conclusions

In [ ]:
print('='*60)
print('  UPES LOST & FOUND — AI MODEL SUMMARY')
print('='*60)
print()
print('  Approach  : TF-IDF Cosine Similarity + Category Boost')
print('              + Color Histogram (Image Matching)')
print()
acc_b, prec_b, rec_b, f1_b, _ = evaluate_model(
    boosted_matrix, lost_items, found_items, ground_truth, threshold=best_t)
print(f'  Accuracy  : {acc_b*100:.1f}%')
print(f'  Precision : {prec_b*100:.1f}%')
print(f'  Recall    : {rec_b*100:.1f}%')
print(f'  F1 Score  : {f1_b*100:.1f}%')
print(f'  Threshold : {best_t:.2f}')
print()
print('  Outputs Generated:')
print('  • text_similarity_matrix.png')
print('  • model_performance.png')
print('  • threshold_analysis.png')
print('  • image_similarity.png')
print('  • combined_scores.png')
print()
print('  Integration: Scores computed by app.py backend')
print('  Notifications sent when score >= threshold')
print('='*60)